::: {.claves}
[En breve · {{< meta tipo >}}]{.etiqueta}

{{< meta resumen >}}
:::

In [1]:
#| label: setup
#| include: false
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
sys.path.insert(0, str(ROOT))

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from common.paths import topic_dir, data_dirs
from common.files import read_processed
from common.plotting import ACCENT, GREY, PALETTE, init_plotly, new_figure, add_source
init_plotly()

# Tablas legibles: separadores en español, sin forma ni tipos
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)
pl.Config.set_thousands_separator(".")
pl.Config.set_decimal_separator(",")
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)

TOPIC = topic_dir()
RAW, PROCESSED = data_dirs(TOPIC)

## Preguntas

Entre 2002 y 2025 la población residente en España pasó de 41,0 a 49,1 millones. Desde 2017 mueren más personas de las que nacen (−116.056 en 2024 [@ine_mnp_2024]), así que la evolución de la población depende de la migración. Pero cuántos «extranjeros» hay cambia según cómo se cuente: este artículo mide la población extranjera de las dos formas y explica por qué difieren.

1. **P1.** ¿Cómo ha evolucionado la población residente en España según su **nacionalidad**, y de dónde procede la población extranjera?
2. **P2.** ¿Cambia el resultado si «extranjero» se define por el **lugar de nacimiento** en lugar de por la nacionalidad? Si cambia, ¿por qué?

## Datos y método

- **Fuente.** INE, Estadística Continua de Población (ECP), tabla 56938 [@ine_ecp_tabla_56938]: población residente a 1 de enero de cada año, de 2002 a 2025, por nacionalidad y país de nacimiento. Datos descargados el 25 de octubre de 2026; metodología en @ine_ecp_metodologia.
- **Unidad de observación: la persona residente.** Es residente quien tiene su residencia habitual en España, entendida por el INE como el lugar de residencia *registrado* [@ine_ecp_metodologia]. La cifra mide personas inscritas como residentes, no permisos de residencia ni movimientos de entrada y salida.
- **Dos formas de definir «español» y «extranjero».**

| Definición | Es español… | Es extranjero… |
|---|---|---|
| **Por nacionalidad** | todo residente con nacionalidad española, haya nacido o no en España | todo residente con nacionalidad extranjera, haya nacido donde haya nacido |
| **Por lugar de nacimiento** | todo residente nacido en España, tenga o no nacionalidad española | todo residente nacido en el extranjero, tenga la nacionalidad que tenga |

  Quien tiene la nacionalidad española y otra se cuenta como español [@ine_ecp_metodologia]. Las regiones de nacionalidad y de nacimiento son las del INE; «Europa» agrupa a la Unión Europea y al resto del continente.

- **Qué se calcula.** Población, peso sobre el total, cambio acumulado desde 2002 (%), media anual de cada periodo (variación dividida entre los años del periodo) y tasa anual compuesta. Las fórmulas están en el *Detalle técnico*.
- **Periodos.** Fijados antes de mirar los datos, por hitos de la economía española: 2002–2008 (expansión), 2008–2014 (crisis), 2014–2019 (recuperación) y 2019–2025 (pandemia y después). Se comprobó que con bloques de 5 años el patrón se mantiene (crecimiento hasta 2008–2010, retroceso hasta 2016–2017 y repunte posterior), aunque cambian las cifras de cada periodo.

In [2]:
#| label: carga
#| code-fold: true
df = read_processed(TOPIC, "poblacion_nacionalidad_origen")

# Series base: todos los sexos y todas las edades
base = df.filter(pl.col("sexo") == "Total", pl.col("edad") == "Todas las edades")


def ancho(d, columna):
    """Tabla ancha: una fila por año (anio) y una columna por categoría."""
    return (d.pivot(on=columna, index="periodo", values="total")
             .sort("periodo")
             .with_columns(anio=pl.col("periodo").dt.year())
             .drop("periodo")
             .select("anio", pl.exclude("anio")))


# P1: por nacionalidad
nac = ancho(base.filter(pl.col("pais_nacimiento") == "Total"), "nacionalidad")

# P2: por país de nacimiento
pais = ancho(base.filter(pl.col("nacionalidad") == "Total"), "pais_nacimiento")

# Cruce nacionalidad x lugar de nacimiento (cuatro grupos)
ESP_ESP = "Nacionalidad española · nacidos en España"
ESP_EXT = "Nacionalidad española · nacidos en el extranjero"
EXT_ESP = "Nacionalidad extranjera · nacidos en España"
EXT_EXT = "Nacionalidad extranjera · nacidos en el extranjero"
cruce = ancho(
    base.filter(pl.col("nacionalidad").is_in(["Española", "Extranjera"]),
                pl.col("pais_nacimiento").is_in(["España", "Extranjero"]))
        .with_columns(grupo=pl.col("nacionalidad") + "|" + pl.col("pais_nacimiento")),
    "grupo",
).rename({"Española|España": ESP_ESP, "Española|Extranjero": ESP_EXT,
          "Extranjera|España": EXT_ESP, "Extranjera|Extranjero": EXT_EXT})
cruce = cruce.with_columns(Total=pl.sum_horizontal(ESP_ESP, ESP_EXT, EXT_ESP, EXT_EXT))

assert nac["anio"].to_list() == pais["anio"].to_list() == cruce["anio"].to_list()
assert (cruce["Total"] - nac["Total"]).abs().max() <= 5

In [3]:
#| label: funciones
#| code-fold: true
PERIODOS = [(2002, 2008), (2008, 2014), (2014, 2019), (2019, 2025)]
PERIODOS_TOTAL = PERIODOS + [(2002, 2025)]
BLOQUES_5 = [(2002, 2007), (2007, 2012), (2012, 2017), (2017, 2022), (2022, 2025)]
ANIOS_CLAVE = [2002, 2008, 2014, 2019, 2025]

REG_NAC = ["Europa", "Sudamérica", "África", "Asia", "América Central y Caribe",
           "América del Norte", "Oceanía", "Apátridas"]
REG_PAIS = ["Europa", "Sudamérica", "África", "Asia", "América Central y Caribe",
            "América del Norte", "Oceanía"]

TINTA = "#4b5563"
COL = {"Española": GREY, "Extranjera": ACCENT, "España": GREY, "Extranjero": ACCENT, "Total": TINTA}
COL_REG = {"Europa": "#2563eb", "Sudamérica": ACCENT, "África": "#0d9488", "Asia": "#a16207",
           "América Central y Caribe": "#7c3aed", "América del Norte": "#db2777",
           "Oceanía": "#65a30d", "Apátridas": GREY}
FUENTE = "Fuente: INE (ECP, tabla 56938), población a 1 de enero. Elaboración propia."


def etiqueta(p):
    return f"{p[0]}–{p[1]}"


def txt_anio(d):
    """Año como texto para mostrarlo en tablas (sin separador de miles)."""
    return d.with_columns(pl.col("anio").cast(pl.String)).rename({"anio": "año"})


def variacion_anual(wide, cols):
    """Variación respecto al año anterior: absoluta (Δ) y en % (Δ%), por columna."""
    return wide.select(
        "anio",
        *[pl.col(c).diff().alias(f"Δ {c}") for c in cols],
        *[(100 * pl.col(c).pct_change()).round(2).alias(f"Δ% {c}") for c in cols],
    )


def resumen_periodos(wide, cols, total, periodos):
    """Por periodo y grupo: población inicial y final, variación, media anual,
    tasa anual compuesta (%) y contribución (%) a la variación del total."""
    idx = {a: i for i, a in enumerate(wide["anio"].to_list())}
    filas = []
    for a, b in periodos:
        var_total = wide[total][idx[b]] - wide[total][idx[a]]
        for c in [*cols, total]:
            p0, p1 = wide[c][idx[a]], wide[c][idx[b]]
            var = p1 - p0
            filas.append({
                "periodo": etiqueta((a, b)), "grupo": c, "inicial": p0, "final": p1,
                "variación": var, "media anual": round(var / (b - a)),
                "tasa compuesta (%)": round(100 * ((p1 / p0) ** (1 / (b - a)) - 1), 2),
                "contribución (%)": round(100 * var / var_total, 1),
            })
    return pl.DataFrame(filas)


def tablas_regiones(wide, regiones, total):
    """Cuatro tablas por región: niveles, peso sobre `total`, media anual y contribución por periodo."""
    clave = wide.filter(pl.col("anio").is_in(ANIOS_CLAVE))
    res = resumen_periodos(wide, regiones, total, PERIODOS_TOTAL)
    res = res.with_columns((100 * (pl.col("final") / pl.col("inicial") - 1)).round(1).alias("variación (%)"))
    orden = {etiqueta(p): i for i, p in enumerate(PERIODOS_TOTAL)}

    def por_periodo(valor):
        return (res.filter(pl.col("grupo") != total)
                   .pivot(on="grupo", index="periodo", values=valor)
                   .with_columns(o=pl.col("periodo").replace_strict(orden)).sort("o").drop("o")
                   .select("periodo", *regiones))

    return {
        "niveles": txt_anio(clave.select("anio", *regiones, total)),
        "peso": txt_anio(clave.select("anio", *[(100 * pl.col(r) / pl.col(total)).round(1).alias(r) for r in regiones])),
        "media": por_periodo("media anual"),
        "tasa": por_periodo("tasa compuesta (%)"),
        "cambio": por_periodo("variación (%)"),
        "contribucion": por_periodo("contribución (%)"),
    }


def fig_total(wide):
    anios = wide["anio"].to_list()
    fig = new_figure(y_title="Variación anual (personas)", height=440)
    pct = (100 * wide["Total"].pct_change()).to_list()
    fig.add_bar(x=anios, y=wide["Total"].diff().to_list(), name="Variación anual (personas y %)", marker_color=GREY,
                text=["" if p is None else ("0,0 %" if round(p, 1) == 0 else f"{p:+.1f} %".replace(".", ",").replace("-", "−")) for p in pct],
                textposition="outside", textfont=dict(size=11), cliponaxis=False, customdata=pct,
                hovertemplate="%{y:,.0f} personas (%{customdata:.2f} %)")
    fig.add_scatter(x=anios, y=wide["Total"].to_list(), name="Población total (eje derecho)",
                    mode="lines+markers", yaxis="y2", line=dict(color=ACCENT, width=3), marker=dict(size=5),
                    hovertemplate="%{y:,.0f}")
    fig.update_layout(yaxis2=dict(overlaying="y", side="right", showgrid=False, tickformat=",.0f",
                                  title="Población total", range=[40e6, 50e6], tick0=40e6, dtick=2e6),
                      yaxis=dict(range=[-300_000, 1_000_000]), xaxis=dict(dtick=2), margin=dict(r=80))
    add_source(fig, FUENTE)
    return fig


def fig_indice_peso(wide, nativo, foraneo, etiqueta_foraneo):
    """Tres paneles: población de cada grupo y total (millones), cambio acumulado desde 2002 (%) y peso del grupo extranjero."""
    anios = wide["anio"].to_list()
    miles = lambda v: f"{v:,.0f}".replace(",", ".")
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                        subplot_titles=("Población (millones)", "Cambio desde 2002 (%)", f"Peso de {etiqueta_foraneo} (%)"))
    y_max = 0
    for g in (nativo, foraneo):
        # panel 1: niveles (millones); etiquetas en el primer y el último año
        mill = (wide[g] / 1e6).to_list()
        fig.add_scatter(x=anios, y=mill, name=g, mode="lines", line=dict(color=COL[g], width=3),
                        text=[f"{miles(p)} personas" for p in wide[g].to_list()], hovertemplate="%{text}", row=1, col=1)
        for i in (0, -1):
            fig.add_annotation(x=anios[i], y=mill[i], text=f"{mill[i]:.1f} M".replace(".", ","), showarrow=False,
                               xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4,
                               yshift=14 if (g == foraneo and i == -1) else -16,
                               font=dict(color=COL[g]), row=1, col=1)
        # panel 2: cambio acumulado (%)
        cambio = (100 * (wide[g] / wide[g][0] - 1)).to_list()
        y_max = max(y_max, max(cambio))
        # texto del hover construido aquí (así el redondeo no depende del formato de Plotly)
        hover = [f"{c:+.0f} % · {miles(p)} personas" for c, p in zip(cambio, wide[g].to_list())]
        fig.add_scatter(x=anios, y=cambio, name=g, mode="lines", line=dict(color=COL[g], width=3), showlegend=False,
                        text=hover, hovertemplate="%{text}", row=1, col=2)
        v = cambio[-1]
        txt = (f"{v:+.1f}" if abs(v) < 10 else f"{v:+.0f}").replace(".", ",")
        fig.add_annotation(x=anios[-1], y=v, text=f"{txt} %", showarrow=False,
                           xanchor="right", yshift=14, font=dict(color=COL[g]), row=1, col=2)
    # panel 1: serie total (referencia)
    mill_t = (wide["Total"] / 1e6).to_list()
    fig.add_scatter(x=anios, y=mill_t, name="Total", mode="lines", line=dict(color=TINTA, width=2, dash="dot"),
                    text=[f"{miles(p)} personas" for p in wide["Total"].to_list()], hovertemplate="%{text}", row=1, col=1)
    for i in (0, -1):
        fig.add_annotation(x=anios[i], y=mill_t[i], text=f"{mill_t[i]:.1f} M".replace(".", ","), showarrow=False,
                           xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4, yshift=26 if i == 0 else 14,
                           font=dict(color=TINTA), row=1, col=1)
    peso = (100 * wide[foraneo] / wide["Total"]).to_list()
    fig.add_scatter(x=anios, y=peso, name=f"Peso de {etiqueta_foraneo}", mode="lines",
                    line=dict(color=ACCENT, width=3), showlegend=False, hovertemplate="%{y:.1f} %", row=1, col=3)
    for i in (0, -1):
        fig.add_annotation(x=anios[i], y=peso[i], text=f"{peso[i]:.1f} %".replace(".", ","), showarrow=False,
                           xanchor="left" if i == 0 else "right", xshift=8 if i == 0 else -4,
                           yshift=-14 if i == 0 else 14, font=dict(color=ACCENT), row=1, col=3)
    fig.update_yaxes(tickformat=".0f", ticksuffix=" M", rangemode="tozero", row=1, col=1)
    fig.update_yaxes(tickformat=",.0f", ticksuffix=" %", zeroline=True, zerolinecolor="rgba(127,127,127,.5)",
                     range=[-0.05 * y_max, 1.12 * y_max], row=1, col=2)
    fig.update_yaxes(tickformat=".0f", ticksuffix=" %", rangemode="tozero", row=1, col=3)
    fig.update_xaxes(dtick=8)
    fig.update_layout(height=440, margin=dict(l=60, r=40, t=110, b=55), hovermode="x unified",
                      legend=dict(y=1.22))
    add_source(fig, FUENTE)
    return fig


def fig_variacion_anual(wide, nativo, foraneo):
    """Barras apiladas de la variación anual (personas); el Δ% de cada grupo (como en la tabla) va en el hover."""
    anios = wide["anio"].to_list()
    fig = new_figure(y_title="Variación anual (personas)", height=440, barmode="relative")

    def pct(g):
        return (100 * wide[g].pct_change()).to_list()

    def fmt(v):
        return "" if v is None or v != v else f"{v:+.2f}".replace(".", ",") + " %"

    for g in (nativo, foraneo):
        d = wide[g].diff().to_list()
        p = [fmt(v) for v in pct(g)]
        hover = [f"{'' if x is None else format(x, '+,.0f').replace(',', '.')} personas ({t})" for x, t in zip(d, p)]
        fig.add_bar(x=anios, y=d, name=g, marker_color=COL[g], hovertext=hover, hovertemplate="%{hovertext}")
    d = wide["Total"].diff().to_list()
    p = [fmt(v) for v in pct("Total")]
    hover = [f"{'' if x is None else format(x, '+,.0f').replace(',', '.')} personas ({t})" for x, t in zip(d, p)]
    fig.add_scatter(x=anios, y=d, name="Total", mode="lines+markers",
                    line=dict(color=TINTA, width=2), marker=dict(size=5), hovertext=hover, hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=1))
    add_source(fig, FUENTE)
    return fig


def fig_periodos(res, grupos, titulo_y="Variación media anual (personas por año)"):
    """Barras apiladas de la media anual por periodo; el rombo es el total. El hover resume media, tasa y contribución."""
    fig = new_figure(y_title=titulo_y, height=440, barmode="relative")
    num = lambda v, f: format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")
    for g in grupos:
        d = res.filter(pl.col("grupo") == g)
        hover = [f"{num(m, '+,.0f')}/año · {num(t, '+.1f')} % anual · {num(c, '.0f')} % del total"
                 for m, t, c in zip(d["media anual"], d["tasa compuesta (%)"], d["contribución (%)"])]
        fig.add_bar(x=d["periodo"].to_list(), y=d["media anual"].to_list(), name=g,
                    marker_color=COL.get(g, COL_REG.get(g)), hovertext=hover, hovertemplate="%{hovertext}")
    t = res.filter(pl.col("grupo") == "Total")
    hover = [f"{format(m, '+,.0f').replace(',', '.')}/año · {format(r, '+.1f').replace('.', ',')} % anual"
             for m, r in zip(t["media anual"], t["tasa compuesta (%)"])]
    fig.add_scatter(x=t["periodo"].to_list(), y=t["media anual"].to_list(), name="Total", mode="markers",
                    marker=dict(symbol="diamond", size=11, color=TINTA, line=dict(color="white", width=1)),
                    hovertext=hover, hovertemplate="%{hovertext}")
    fig.update_xaxes(type="category")
    add_source(fig, FUENTE)
    return fig


def fig_regiones(wide, regiones, total, nombre_total):
    """Tres vistas con botones: personas, peso (%) y variación media anual por periodo."""
    anios = wide["anio"].to_list()
    res = resumen_periodos(wide, regiones, total, PERIODOS_TOTAL)
    n = len(regiones)
    fig = new_figure(y_title="Personas", height=520, barmode="relative")
    for r in regiones:
        fig.add_scatter(x=anios, y=wide[r].to_list(), name=r, mode="lines",
                        line=dict(color=COL_REG[r], width=2.5), hovertemplate="%{y:,.0f}")
    for r in regiones:
        fig.add_scatter(x=anios, y=(100 * wide[r] / wide[total]).to_list(), name=r, mode="lines",
                        line=dict(color=COL_REG[r], width=2.5), hovertemplate="%{y:.1f} %", visible=False)
    tot = res.filter(pl.col("grupo") == total)
    num = lambda v, f: format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")
    for r in regiones:
        d = res.filter(pl.col("grupo") == r)
        hover = [f"{num(m, '+,.0f')}/año · {num(c, '.0f')} % del total"
                 for m, c in zip(d["media anual"], d["contribución (%)"])]
        fig.add_bar(x=d["periodo"].to_list(), y=d["media anual"].to_list(), name=r, marker_color=COL_REG[r],
                    hovertext=hover, hovertemplate="%{hovertext}", visible=False)
    fig.add_scatter(x=tot["periodo"].to_list(), y=tot["media anual"].to_list(), name=nombre_total.capitalize(),
                    mode="markers", marker=dict(symbol="diamond", size=10, color=TINTA, line=dict(color="white", width=1)),
                    hovertext=[f"{num(t, '+,.0f')}/año (= 100 %)" for t in tot["media anual"]],
                    hovertemplate="%{hovertext}", visible=False)

    def vista(k, titulo, sufijo, formato, tipo_x):
        dtick = 2 if tipo_x == "linear" else 1
        visible = [i // n == k for i in range(3 * n)] + [k == 2]
        return dict(label=["Personas", "Peso (%)", "Variación media anual por periodo"][k], method="update",
                    args=[{"visible": visible},
                          {"yaxis.title.text": titulo, "yaxis.ticksuffix": sufijo, "yaxis.tickformat": formato,
                           "xaxis.type": tipo_x, "xaxis.dtick": dtick}])

    botones = [vista(0, "Personas", "", ",.0f", "linear"),
               vista(1, f"% sobre {nombre_total}", " %", ".1f", "linear"),
               vista(2, "Personas por año", "", ",.0f", "category")]
    fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.3, xanchor="left",
                                        yanchor="bottom", buttons=botones, showactive=True)],
                      margin=dict(t=120), xaxis=dict(dtick=2))
    add_source(fig, FUENTE)
    return fig


COL.update({"Nacionalidad española · nacidos en España": GREY, "Nacionalidad española · nacidos en el extranjero": "#2563eb",
            "Nacionalidad extranjera · nacidos en España": "#0d9488", "Nacionalidad extranjera · nacidos en el extranjero": ACCENT})


def _es(v, f):
    """Formato español (miles con punto, decimales con coma) construido a mano para el hover."""
    return format(v, f).replace(",", "X").replace(".", ",").replace("X", ".")


def fig_cruce_pesos(cr):
    """Barras apiladas: peso (%) sobre el total de los tres grupos que no son «nacionalidad española · nacidos en España»."""
    anios = cr["anio"].to_list()
    fig = new_figure(y_title="% sobre la población total", height=440, barmode="stack")
    for g in (ESP_EXT, EXT_ESP, EXT_EXT):
        pesos = (100 * cr[g] / cr["Total"]).to_list()
        fig.add_bar(x=anios, y=pesos, name=g, marker_color=COL[g],
                    hovertext=[f"{_es(p, '.1f')} % · {_es(n, ',.0f')} personas" for p, n in zip(pesos, cr[g].to_list())],
                    hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=2))
    fig.update_yaxes(ticksuffix=" %", tickformat=".0f")
    add_source(fig, FUENTE)
    return fig


def fig_definiciones(cr):
    """Dos paneles: población extranjera según cada definición (millones) y su peso sobre el total (%)."""
    anios = cr["anio"].to_list()
    series = {"Residentes con nacionalidad extranjera": cr[EXT_ESP] + cr[EXT_EXT],
              "Residentes nacidos en el extranjero": cr[ESP_EXT] + cr[EXT_EXT]}
    colores = {"Residentes con nacionalidad extranjera": ACCENT, "Residentes nacidos en el extranjero": "#2563eb"}
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=("Población «extranjera» según cada definición (millones)", "Peso sobre la población total (%)"))
    for nombre, ser in series.items():
        mill = (ser / 1e6).to_list()
        peso = (100 * ser / cr["Total"]).to_list()
        fig.add_scatter(x=anios, y=mill, name=nombre, mode="lines", line=dict(color=colores[nombre], width=3),
                        hovertext=[f"{_es(n, ',.0f')} personas" for n in ser.to_list()], hovertemplate="%{hovertext}", row=1, col=1)
        fig.add_scatter(x=anios, y=peso, name=nombre, mode="lines", line=dict(color=colores[nombre], width=3),
                        showlegend=False, hovertext=[f"{_es(p, '.1f')} %" for p in peso], hovertemplate="%{hovertext}", row=1, col=2)
        for col, y, txt in ((1, mill[-1], f"{_es(mill[-1], '.1f')} M"), (2, peso[-1], f"{_es(peso[-1], '.1f')} %")):
            fig.add_annotation(x=anios[-1], y=y, text=txt, showarrow=False, xanchor="right", yshift=14,
                               font=dict(color=colores[nombre]), row=1, col=col)
    fig.update_yaxes(rangemode="tozero", ticksuffix=" M", tickformat=".0f", row=1, col=1)
    fig.update_yaxes(rangemode="tozero", ticksuffix=" %", tickformat=".0f", row=1, col=2)
    fig.update_xaxes(dtick=4)
    fig.update_layout(height=440, margin=dict(l=60, r=40, t=110, b=55), hovermode="x unified", legend=dict(y=1.22))
    add_source(fig, FUENTE)
    return fig


def fig_explica(cr):
    """Cuatro paneles (2 x 2): las proporciones que explican la diferencia entre ambas definiciones."""
    anios = cr["anio"].to_list()
    paneles = [
        ("Nacionalidad española y nacidos fuera<br>(% de los de nacionalidad española)", cr[ESP_EXT], cr[ESP_ESP] + cr[ESP_EXT], COL[ESP_EXT]),
        ("Nacionalidad extranjera y nacidos en España<br>(% de los de nacionalidad extranjera)", cr[EXT_ESP], cr[EXT_ESP] + cr[EXT_EXT], COL[EXT_ESP]),
        ("Nacidos fuera con nacionalidad extranjera<br>(% de los nacidos fuera)", cr[EXT_EXT], cr[ESP_EXT] + cr[EXT_EXT], COL[EXT_EXT]),
        ("Nacidos fuera con nacionalidad española<br>(% de los nacidos fuera)", cr[ESP_EXT], cr[ESP_EXT] + cr[EXT_EXT], COL[ESP_EXT]),
    ]
    fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.1, vertical_spacing=0.2,
                        subplot_titles=[p[0] for p in paneles])
    for k, (_, num, den, color) in enumerate(paneles):
        r, c = k // 2 + 1, k % 2 + 1
        p = (100 * num / den).to_list()
        fig.add_scatter(x=anios, y=p, mode="lines", line=dict(color=color, width=3), showlegend=False,
                        hovertext=[f"{_es(v, '.1f')} % · {_es(n, ',.0f')} de {_es(d, ',.0f')}"
                                   for v, n, d in zip(p, num.to_list(), den.to_list())],
                        hovertemplate="%{hovertext}", row=r, col=c)
        for i in (0, -1):
            fig.add_annotation(x=anios[i], y=p[i], text=f"{_es(p[i], '.1f')} %", showarrow=False,
                               xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4,
                               yshift=-14 if (i == 0 or p[-1] < p[-2]) else 14, font=dict(color=color), row=r, col=c)
        fig.update_yaxes(rangemode="tozero", ticksuffix=" %", tickformat=".0f", row=r, col=c)
    fig.update_xaxes(dtick=4)
    fig.update_layout(height=720, margin=dict(l=60, r=40, t=80, b=55), hovermode="x unified")
    for a in fig.layout.annotations[:4]:
        a.font.size = 12
    add_source(fig, FUENTE)
    return fig


def fig_composicion(wide, regiones, total, titulo_y):
    """Barras apiladas: peso (%) de cada región sobre el total del grupo."""
    anios = wide["anio"].to_list()
    fig = new_figure(y_title=titulo_y, height=440, barmode="stack")
    for r in regiones:
        pesos = (100 * wide[r] / wide[total]).to_list()
        fig.add_bar(x=anios, y=pesos, name=r, marker_color=COL_REG[r],
                    hovertext=[f"{_es(p, '.1f')} % · {_es(n, ',.0f')} personas" for p, n in zip(pesos, wide[r].to_list())],
                    hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=2))
    fig.update_yaxes(ticksuffix=" %", tickformat=".0f", range=[0, 100])
    add_source(fig, FUENTE)
    return fig


def fig_region_doble(nac, pais, region, etiqueta_nac, etiqueta_nac_pais):
    """Una región medida de dos maneras: por nacionalidad y por lugar de nacimiento (millones de personas)."""
    anios = nac["anio"].to_list()
    fig = new_figure(y_title="Millones de personas", height=420)
    for k, (ser, nombre, color) in enumerate(((pais[region], etiqueta_nac_pais, "#2563eb"), (nac[region], etiqueta_nac, ACCENT))):
        mill = (ser / 1e6).to_list()
        fig.add_scatter(x=anios, y=mill, name=nombre, mode="lines", line=dict(color=color, width=3),
                        hovertext=[f"{_es(n, ',.0f')} personas" for n in ser.to_list()], hovertemplate="%{hovertext}")
        for i in (0, -1):
            fig.add_annotation(x=anios[i], y=mill[i], text=f"{_es(mill[i], '.1f')} M", showarrow=False,
                               xanchor="left" if i == 0 else "right", xshift=6 if i == 0 else -4,
                               yshift=-16 if (k == 1 and i == 0) else 14, font=dict(color=color))
    fig.update_layout(xaxis=dict(dtick=2))
    fig.update_yaxes(rangemode="tozero", ticksuffix=" M", tickformat=".0f", dtick=1)
    add_source(fig, FUENTE)
    return fig


def fig_esp_ext_anual(cr):
    """Barras: aumento anual de los residentes con nacionalidad española nacidos en el extranjero (hover: personas y %)."""
    anios = cr["anio"].to_list()[1:]
    d = cr[ESP_EXT].diff().to_list()[1:]
    p = (100 * cr[ESP_EXT].pct_change()).to_list()[1:]
    fig = new_figure(y_title="Aumento anual (personas)", height=400)
    fig.add_bar(x=anios, y=d, name=ESP_EXT, marker_color=COL[ESP_EXT],
                hovertext=[f"{_es(v, '+,.0f')} personas ({_es(q, '+.1f')} %)" for v, q in zip(d, p)],
                hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=2), showlegend=False)
    fig.update_yaxes(tickformat=",.0f")
    add_source(fig, FUENTE)
    return fig


def fig_aumento_anual(cr):
    """Barras agrupadas: aumento anual de la población extranjera según cada definición."""
    anios = cr["anio"].to_list()[1:]
    series = {"Con nacionalidad extranjera": cr[EXT_ESP] + cr[EXT_EXT], "Nacidos en el extranjero": cr[ESP_EXT] + cr[EXT_EXT]}
    colores = {"Con nacionalidad extranjera": ACCENT, "Nacidos en el extranjero": "#2563eb"}
    fig = new_figure(y_title="Aumento anual (personas)", height=420, barmode="group")
    for nombre, ser in series.items():
        d = ser.diff().to_list()[1:]
        p = (100 * ser.pct_change()).to_list()[1:]
        fig.add_bar(x=anios, y=d, name=nombre, marker_color=colores[nombre],
                    hovertext=[f"{_es(v, '+,.0f')} personas ({_es(q, '+.1f')} %)" for v, q in zip(d, p)],
                    hovertemplate="%{hovertext}")
    fig.update_layout(xaxis=dict(dtick=2))
    fig.update_yaxes(tickformat=",.0f")
    add_source(fig, FUENTE)
    return fig


def fig_esp_esp(cr):
    """Dos paneles: residentes con nacionalidad española nacidos en España (millones) y su peso sobre el total (%)."""
    anios = cr["anio"].to_list()
    mill = (cr[ESP_ESP] / 1e6).to_list()
    peso = (100 * cr[ESP_ESP] / cr["Total"]).to_list()
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=("Personas (millones)", "Peso sobre la población total (%)"))
    fig.add_scatter(x=anios, y=mill, mode="lines", line=dict(color=GREY, width=3), showlegend=False,
                    hovertext=[f"{_es(n, ',.0f')} personas" for n in cr[ESP_ESP].to_list()], hovertemplate="%{hovertext}", row=1, col=1)
    fig.add_scatter(x=anios, y=peso, mode="lines", line=dict(color=GREY, width=3), showlegend=False,
                    hovertext=[f"{_es(v, '.1f')} %" for v in peso], hovertemplate="%{hovertext}", row=1, col=2)
    i_max = max(range(len(mill)), key=lambda k: mill[k])
    for k, pos in ((0, "left"), (i_max, "center"), (len(mill) - 1, "right")):
        fig.add_annotation(x=anios[k], y=mill[k], text=f"{_es(mill[k], '.2f')} M", showarrow=False, xanchor=pos,
                           yshift=14, font=dict(color=TINTA), row=1, col=1)
    for k, pos in ((0, "left"), (len(peso) - 1, "right")):
        fig.add_annotation(x=anios[k], y=peso[k], text=f"{_es(peso[k], '.1f')} %", showarrow=False, xanchor=pos,
                           yshift=14, font=dict(color=TINTA), row=1, col=2)
    # ejes desde cero para no exagerar visualmente los cambios
    fig.update_yaxes(ticksuffix=" M", tickformat=".0f", range=[0, 1.12 * max(mill)], row=1, col=1)
    fig.update_yaxes(ticksuffix=" %", tickformat=".0f", range=[0, 105], row=1, col=2)
    fig.update_xaxes(dtick=4)
    fig.update_layout(height=420, margin=dict(l=60, r=40, t=90, b=55), hovermode="x unified")
    add_source(fig, FUENTE)
    return fig

## Resultados

### 1. La población con nacionalidad extranjera explica la mayor parte del crecimiento

**Qué se mira.** La población con nacionalidad española y extranjera, cuánto ha cambiado cada una desde 2002 y qué parte del total es extranjera. **Cómo leer el gráfico:** el panel izquierdo muestra los niveles (millones de personas; la línea punteada es el total); el central, el cambio acumulado desde 2002 (0 % es el valor de 2002 y +100 % el doble); el derecho, el peso de la población extranjera sobre el total.

In [4]:
#| label: tbl-resumen-nac
#| tbl-cap: "Población por nacionalidad, peso de la población extranjera y cambio acumulado desde 2002"
(nac.filter(pl.col("anio").is_in(ANIOS_CLAVE))
    .select("anio", "Española", "Extranjera", "Total",
            (100 * pl.col("Extranjera") / pl.col("Total")).round(1).alias("% extranjera"),
            (100 * (pl.col("Española") - nac["Española"][0]) / nac["Española"][0]).round(1).alias("Cambio española (%)"),
            (100 * (pl.col("Extranjera") - nac["Extranjera"][0]) / nac["Extranjera"][0]).round(1).alias("Cambio extranjera (%)"))
    .pipe(txt_anio))

año,Española,Extranjera,Total,% extranjera,Cambio española (%),Cambio extranjera (%)
"""2002""",39.297.299,1.737.972,41.035.271,"4,2","0,0","0,0"
"""2008""",40.582.643,5.086.295,45.668.938,"11,1","3,3","192,7"
"""2014""",41.819.392,4.676.352,46.495.744,"10,1","6,4","169,1"
"""2019""",42.068.189,4.850.762,46.918.951,"10,3","7,1","179,1"
"""2025""",42.216.326,6.911.971,49.128.297,"14,1","7,4","297,7"


In [5]:
#| label: fig-peso-nac
#| fig-cap: "Izquierda: población española, extranjera y total (millones). Centro: cambio acumulado desde 2002 (%). Derecha: peso de la población extranjera sobre el total (%)"
fig_indice_peso(nac, "Española", "Extranjera", "la población extranjera").show()

**Lectura.** Los datos (@tbl-resumen-nac) y el gráfico (@fig-peso-nac) muestran dos trayectorias muy distintas.

- **Total y aportación.** La población pasa de 41,0 a 49,1 millones (+8,1 millones). La población con nacionalidad extranjera explica el 63,9 % de ese aumento (+5,17 millones); la española, el 36,1 % (+2,92 millones). *(Panel izquierdo y tabla.)*
- **Población extranjera.** Pasa de 1,74 a 6,91 millones: +298 % desde 2002, casi cuatro veces más. *(Panel central: línea naranja.)*
- **Población española.** Crece un 7,4 % (de 39,30 a 42,22 millones) y se mueve entre 42,0 y 42,2 millones desde 2016. *(Panel central: línea gris.)*
- **Peso.** La población extranjera pasa del 4,2 % del total en 2002 al 11,1 % en 2008, el 10,1 % en 2014, el 10,3 % en 2019 y el 14,1 % en 2025. El máximo anterior a 2025 fue el 11,6 % (2009). *(Panel derecho.)*

**Aprendizaje.** Casi dos tercios del aumento de población de estos 23 años corresponden a la población con nacionalidad extranjera, pero su peso no ha subido de forma continua: el peso baja entre 2009 y 2016–2017 antes de alcanzar su máximo en 2025.

### 2. El ritmo no ha sido constante: cuatro fases

**Qué se mira.** La variación media anual de cada grupo en los cuatro periodos. **Cómo leer el gráfico:** cada tramo de color es la media anual del grupo (personas que gana o pierde cada año, de media); el rombo, la del total.

In [6]:
#| label: tbl-periodos-nac
#| tbl-cap: "Variación media anual (personas por año) y tasa anual compuesta (%) por periodo, según nacionalidad"
res_nac = resumen_periodos(nac, ["Española", "Extranjera"], "Total", PERIODOS_TOTAL)
(res_nac.pivot(on="grupo", index="periodo", values="media anual")
        .rename({"Española": "Española (media anual)", "Extranjera": "Extranjera (media anual)", "Total": "Total (media anual)"})
        .join(res_nac.filter(pl.col("grupo") == "Extranjera").select("periodo", pl.col("tasa compuesta (%)").alias("Extranjera (tasa anual, %)")),
              on="periodo"))

periodo,Española (media anual),Extranjera (media anual),Total (media anual),"Extranjera (tasa anual, %)"
"""2002–2008""",214.224,558.054,772.278,"19,6"
"""2008–2014""",206.125,-68.324,137.801,"-1,39"
"""2014–2019""",49.759,34.882,84.641,"0,74"
"""2019–2025""",24.690,343.535,368.224,"6,08"
"""2002–2025""",126.914,224.956,351.871,"6,19"


In [7]:
#| label: fig-periodos-nac
#| fig-cap: "Variación media anual por periodo, según nacionalidad (cada tramo es la media anual del grupo; el rombo, la del total)"
fig_periodos(res_nac.filter(pl.col("periodo") != "2002–2025"), ["Española", "Extranjera"]).show()

**Lectura.** La tabla (@tbl-periodos-nac) y las barras (@fig-periodos-nac) separan cuatro fases.

- **2002–2008.** La población con nacionalidad extranjera crece 558.054 personas al año (19,6 % anual); la española, 214.224. El total crece 772.278 al año.
- **2008–2014.** La población con nacionalidad extranjera cae 68.324 al año (−1,39 % anual) mientras la española sigue creciendo (+206.125 al año). El total sube solo 137.801 al año.
- **2014–2019.** Crecimiento débil de ambas: +34.882 (extranjera) y +49.759 (española) al año.
- **2019–2025.** La población extranjera crece 343.535 al año (6,08 % anual) y explica el 93,3 % del aumento del total (368.224 al año); la española, 24.690.

**Aprendizaje.** El crecimiento se concentra en dos periodos (2002–2008 y 2019–2025). Entre ambos la población con nacionalidad extranjera se estanca o baja, y es la española la que sostiene el aumento. La sección 5 (@tbl-esp-composicion) muestra que en 2008–2014 el 54 % del aumento de la población con nacionalidad española (+664.494 de +1.236.748) son nacidos en el extranjero.

### 3. ¿De dónde procede la población extranjera?

**Qué se mira.** Cuántas personas tienen la nacionalidad de cada región y qué peso tiene cada región sobre el total de la población extranjera. **Cómo leer el gráfico:** cada barra suma 100 %; la altura de cada color es el peso de esa región ese año.

In [8]:
#| label: tbl-peso-region
#| tbl-cap: "Peso de cada región de nacionalidad sobre la población extranjera (%)"
txt_anio(nac.filter(pl.col("anio").is_in(ANIOS_CLAVE)).select(
    "anio", *[(100 * pl.col(r) / pl.col("Extranjera")).round(1).alias(r) for r in REG_NAC]))

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas
"""2002""","34,0","34,7","21,0","4,6","3,9","1,6","0,1","0,0"
"""2008""","42,5","30,0","18,0","5,2","3,2","1,0","0,1","0,0"
"""2014""","46,5","19,4","21,4","7,3","4,2","1,1","0,1","0,0"
"""2019""","43,7","18,4","22,0","8,9","5,7","1,3","0,1","0,1"
"""2025""","35,4","28,3","19,8","8,1","6,6","1,5","0,1","0,1"


In [9]:
#| label: tbl-region-niveles
#| tbl-cap: "Población con nacionalidad extranjera por región de nacionalidad (personas)"
txt_anio(nac.filter(pl.col("anio").is_in(ANIOS_CLAVE)).select("anio", *REG_NAC, "Extranjera"))

año,Europa,Sudamérica,África,Asia,América Central y Caribe,América del Norte,Oceanía,Apátridas,Extranjera
"""2002""",590.357,603.294,364.732,80.773,68.392,28.530,1.506,389,1.737.972
"""2008""",2.161.986,1.525.069,917.729,264.904,164.081,49.285,2.615,626,5.086.295
"""2014""",2.175.886,907.577,1.002.138,341.763,195.742,49.884,2.437,925,4.676.352
"""2019""",2.118.928,891.421,1.066.160,432.783,274.911,61.078,2.944,2.537,4.850.762
"""2025""",2.449.192,1.958.005,1.367.584,563.293,459.376,104.691,5.049,4.781,6.911.971


In [10]:
#| label: fig-composicion-nac
#| fig-cap: "Peso de cada región de nacionalidad sobre la población extranjera (%)"
fig_composicion(nac, REG_NAC, "Extranjera", "% sobre la población extranjera").show()

**Lectura.** Las tablas de personas (@tbl-region-niveles) y de pesos (@tbl-peso-region) y las barras (@fig-composicion-nac) muestran la composición por región de nacionalidad.

- **Europa** pasa del 34,0 % (2002) al 42,5 % (2008) y al 46,5 % (2014), y baja al 35,4 % en 2025 (2,45 millones). Es la región que más crece en 2002–2008: +1.571.629 personas de las +3.348.323 que aumenta la población extranjera (el 46,9 %).
- **Sudamérica** pasa del 34,7 % al 30,0 % (2008) y al 19,4 % (2014); en 2025 vuelve al 28,3 % (1,96 millones).
- **África** se mantiene entre el 18,0 % y el 22,0 % en los años de la tabla (1,37 millones en 2025).
- **Asia** (del 4,6 % al 8,1 %) y **América Central y Caribe** (del 3,9 % al 6,6 %) casi duplican su peso.

**Aprendizaje.** La procedencia cambia por fases: Europa es la mayor región en todos los años de la tabla salvo 2002; Sudamérica pierde 10,6 puntos entre 2008 y 2014 y recupera 9,9 entre 2019 y 2025. Esa caída de Sudamérica es la que la sección 6 examina con las dos definiciones.

### 4. «Extranjero» depende de cómo se defina

**Qué se mira.** El tamaño de la población extranjera según las dos definiciones de la sección «Datos y método». **Cómo leer el gráfico:** el panel izquierdo da millones de personas y el derecho su peso sobre el total; la distancia entre las dos líneas es la diferencia entre definiciones.

In [11]:
#| label: tbl-definiciones
#| tbl-cap: "Dos definiciones de «extranjero»: residentes con nacionalidad extranjera y residentes nacidos en el extranjero (personas y % de la población total)"
d = cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE))
txt_anio(d.select(
    "anio",
    (pl.col(EXT_ESP) + pl.col(EXT_EXT)).alias("Con nacionalidad extranjera"),
    (pl.col(ESP_EXT) + pl.col(EXT_EXT)).alias("Nacidos en el extranjero"),
    (pl.col(ESP_EXT) - pl.col(EXT_ESP)).alias("Diferencia (personas)"),
    (100 * (pl.col(EXT_ESP) + pl.col(EXT_EXT)) / pl.col("Total")).round(1).alias("% con nacionalidad extranjera"),
    (100 * (pl.col(ESP_EXT) + pl.col(EXT_EXT)) / pl.col("Total")).round(1).alias("% nacidos en el extranjero"),
    (100 * (pl.col(ESP_EXT) - pl.col(EXT_ESP)) / pl.col("Total")).round(1).alias("Diferencia (p. p.)"),
))

año,Con nacionalidad extranjera,Nacidos en el extranjero,Diferencia (personas),% con nacionalidad extranjera,% nacidos en el extranjero,Diferencia (p. p.)
"""2002""",1.737.972,2.334.098,596.126,"4,2","5,7","1,5"
"""2008""",5.086.295,5.878.920,792.625,"11,1","12,9","1,7"
"""2014""",4.676.352,5.952.770,1.276.418,"10,1","12,8","2,7"
"""2019""",4.850.762,6.549.309,1.698.547,"10,3","14,0","3,6"
"""2025""",6.911.971,9.464.210,2.552.239,"14,1","19,3","5,2"


In [12]:
#| label: fig-definiciones
#| fig-cap: "Población «extranjera» según se defina por nacionalidad o por lugar de nacimiento. Izquierda: millones de personas. Derecha: peso sobre la población total (%)"
fig_definiciones(cruce).show()

In [13]:
#| label: tbl-definiciones-periodos
#| tbl-cap: "Variación de la población extranjera por periodo según cada definición: personas, % y aportación al aumento del total"
idx = {a: k for k, a in enumerate(cruce["anio"].to_list())}
nac_ext = cruce[EXT_ESP] + cruce[EXT_EXT]
nac_fuera = cruce[ESP_EXT] + cruce[EXT_EXT]
filas = []
for a, b in PERIODOS_TOTAL:
    vt = cruce["Total"][idx[b]] - cruce["Total"][idx[a]]
    v1, v2 = nac_ext[idx[b]] - nac_ext[idx[a]], nac_fuera[idx[b]] - nac_fuera[idx[a]]
    filas.append({"periodo": etiqueta((a, b)), "Total (personas)": vt,
                  "Nac. extranjera (personas)": v1, "Nac. extranjera (%)": round(100 * v1 / nac_ext[idx[a]], 1),
                  "Nac. extranjera (% del aumento del total)": round(100 * v1 / vt, 1),
                  "Nacidos fuera (personas)": v2, "Nacidos fuera (%)": round(100 * v2 / nac_fuera[idx[a]], 1),
                  "Nacidos fuera (% del aumento del total)": round(100 * v2 / vt, 1)})
pl.DataFrame(filas)

periodo,Total (personas),Nac. extranjera (personas),Nac. extranjera (%),Nac. extranjera (% del aumento del total),Nacidos fuera (personas),Nacidos fuera (%),Nacidos fuera (% del aumento del total)
"""2002–2008""",4.633.668,3.348.323,"192,7","72,3",3.544.822,"151,9","76,5"
"""2008–2014""",826.805,-409.943,"-8,1","-49,6",73.850,"1,3","8,9"
"""2014–2019""",423.207,174.410,"3,7","41,2",596.539,"10,0","141,0"
"""2019–2025""",2.209.346,2.061.209,"42,5","93,3",2.914.901,"44,5","131,9"
"""2002–2025""",8.093.026,5.173.999,"297,7","63,9",7.130.112,"305,5","88,1"


In [14]:
#| label: tbl-serie-anual
#| tbl-cap: "Serie anual: población total y población extranjera según cada definición, con su aumento respecto al año anterior (personas)"
txt_anio(cruce.select(
    "anio", "Total", pl.col("Total").diff().alias("Aumento total"),
    (pl.col(EXT_ESP) + pl.col(EXT_EXT)).alias("Nac. extranjera"),
    (pl.col(EXT_ESP) + pl.col(EXT_EXT)).diff().alias("Aumento nac. extranjera"),
    (pl.col(ESP_EXT) + pl.col(EXT_EXT)).alias("Nacidos fuera"),
    (pl.col(ESP_EXT) + pl.col(EXT_EXT)).diff().alias("Aumento nacidos fuera")).drop_nulls())

año,Total,Aumento total,Nac. extranjera,Aumento nac. extranjera,Nacidos fuera,Aumento nacidos fuera
"""2003""",41.827.835,792.564,2.362.027,624.055,2.984.094,649.996
"""2004""",42.547.454,719.619,2.900.787,538.760,3.547.669,563.575
"""2005""",43.296.335,748.881,3.430.204,529.417,4.107.226,559.557
"""2006""",44.009.968,713.633,3.930.916,500.712,4.637.971,530.745
"""2007""",44.784.659,774.691,4.449.434,518.518,5.200.062,562.091
"""2008""",45.668.939,884.280,5.086.295,636.861,5.878.920,678.858
"""2009""",46.239.270,570.331,5.386.659,300.364,6.225.513,346.593
"""2010""",46.486.621,247.351,5.402.579,15.920,6.280.064,54.551
"""2011""",46.667.176,180.555,5.312.441,-90.138,6.282.209,2.145
"""2012""",46.818.216,151.040,5.236.030,-76.411,6.294.952,12.743


In [15]:
#| label: fig-aumento-anual
#| fig-cap: "Aumento anual de la población extranjera según cada definición (personas; al pasar el cursor, también en %)"
fig_aumento_anual(cruce).show()

**Lectura.** La tabla (@tbl-definiciones), el gráfico (@fig-definiciones), la tabla por periodos (@tbl-definiciones-periodos) y la serie anual (@tbl-serie-anual, @fig-aumento-anual) comparan ambas definiciones.

- **Niveles.** La definición por lugar de nacimiento da siempre una cifra mayor: 2.334.098 frente a 1.737.972 en 2002 (5,7 % frente a 4,2 %) y 9.464.210 frente a 6.911.971 en 2025 (19,3 % frente a 14,1 %).
- **Diferencia.** Pasa de 596.126 personas (1,5 puntos porcentuales) en 2002 a 2.552.239 (5,2 puntos) en 2025. Crece todos los años.
- **La crisis.** En 2008–2014 la población con nacionalidad extranjera cae 409.943 personas (−8,1 %) mientras la nacida en el extranjero sube 73.850 (+1,3 %). En 2014–2019 las dos crecen, pero la segunda más (+10,0 % frente a +3,7 %).
- **2019–2025.** Las dos crecen a un ritmo parecido: +42,5 % con nacionalidad extranjera y +44,5 % nacidos en el extranjero.
- **Aportación al aumento del total** (@tbl-definiciones-periodos, fila 2002–2025). La población con nacionalidad extranjera explica el 63,9 % del aumento de 8.093.026 personas; la nacida en el extranjero, el 88,1 %. Desde 2014 la nacida fuera aumenta más que el total (141,0 % del aumento en 2014–2019 y 131,9 % en 2019–2025) porque la nacida en España disminuye.
- **Año a año** (@tbl-serie-anual, @fig-aumento-anual). La población total supera los 47 millones desde 2020 (47.318.050) y llega a 49.128.297 en 2025. Los mayores aumentos anuales de la población con nacionalidad extranjera son los de 2008 (+636.861) y 2003 (+624.055), por delante de 2023 (+580.574). Por lugar de nacimiento, el mayor es el de 2023 (+736.090), por delante de 2008 (+678.858).
- **Últimos años.** Entre 2018 y 2025 la población con nacionalidad extranjera aumenta en 2.334.649 personas y la nacida en el extranjero, en 3.256.701. Entre 2021 y 2025, en 1.509.269 y 2.209.413.

**Aprendizaje.** La definición no solo cambia el nivel: cambia la historia, e incluso cuál es el año de mayor crecimiento (2008 por nacionalidad, 2023 por lugar de nacimiento). Con la nacionalidad, la crisis de 2008 reduce la población extranjera; con el lugar de nacimiento, entre 2008 y 2014 queda casi igual (con un descenso moderado entre 2012 y 2015, visible en el panel izquierdo).

### 5. Qué explica la diferencia

**Qué se mira.** La diferencia entre definiciones es igual a los residentes con nacionalidad española nacidos en el extranjero menos los residentes con nacionalidad extranjera nacidos en España. **Cómo leer el gráfico:** cada panel es una proporción distinta (el título dice de qué grupo es porcentaje).

In [16]:
#| label: tbl-explica
#| tbl-cap: "Los dos grupos que explican la diferencia: personas y % dentro de su grupo de nacionalidad"
txt_anio(cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE)).select(
    "anio",
    pl.col(ESP_EXT).alias("Nacionalidad española, nacidos fuera"),
    (100 * pl.col(ESP_EXT) / (pl.col(ESP_ESP) + pl.col(ESP_EXT))).round(1).alias("% de los de nacionalidad española"),
    pl.col(EXT_ESP).alias("Nacionalidad extranjera, nacidos en España"),
    (100 * pl.col(EXT_ESP) / (pl.col(EXT_ESP) + pl.col(EXT_EXT))).round(1).alias("% de los de nacionalidad extranjera"),
    (100 * pl.col(ESP_EXT) / (pl.col(ESP_EXT) + pl.col(EXT_EXT))).round(1).alias("% de los nacidos fuera con nacionalidad española"),
))

año,"Nacionalidad española, nacidos fuera",% de los de nacionalidad española,"Nacionalidad extranjera, nacidos en España",% de los de nacionalidad extranjera,% de los nacidos fuera con nacionalidad española
"""2002""",675.371,"1,7",79.245,"4,6","28,9"
"""2008""",1.045.500,"2,6",252.875,"5,0","17,8"
"""2014""",1.709.994,"4,1",433.576,"9,3","28,7"
"""2019""",2.215.131,"5,3",516.584,"10,6","33,8"
"""2025""",3.051.382,"7,2",499.143,"7,2","32,2"


In [17]:
#| label: tbl-esp-esp
#| tbl-cap: "Residentes con nacionalidad española nacidos en España: personas y peso sobre la población total"
txt_anio(cruce.filter(pl.col("anio").is_in(sorted(ANIOS_CLAVE + [2012]))).select(
    "anio", pl.col(ESP_ESP).alias("Personas"), (100 * pl.col(ESP_ESP) / pl.col("Total")).round(1).alias("% del total")))

año,Personas,% del total
"""2002""",38.621.928,"94,1"
"""2008""",39.537.144,"86,6"
"""2012""",40.130.756,"85,7"
"""2014""",40.109.398,"86,3"
"""2019""",39.853.058,"84,9"
"""2025""",39.164.944,"79,7"


In [18]:
#| label: fig-esp-esp
#| fig-cap: "Residentes con nacionalidad española nacidos en España. Izquierda: millones de personas (con el máximo de la serie). Derecha: peso sobre la población total (%)"
fig_esp_esp(cruce).show()

In [19]:
#| label: tbl-esp-composicion
#| tbl-cap: "Aumento de la población con nacionalidad española por periodo, según haya nacido en España o en el extranjero (personas)"
(cruce.filter(pl.col("anio").is_in(ANIOS_CLAVE))
      .select("anio", pl.col(ESP_ESP), pl.col(ESP_EXT))
      .with_columns(periodo=pl.format("{}–{}", pl.col("anio").shift(1), pl.col("anio")),
                    total=(pl.col(ESP_ESP) + pl.col(ESP_EXT)).diff(),
                    en_espana=pl.col(ESP_ESP).diff(), fuera=pl.col(ESP_EXT).diff())
      .drop_nulls("periodo")
      .select("periodo", pl.col("total").alias("Aumento con nacionalidad española"),
              pl.col("en_espana").alias("Nacidos en España"), pl.col("fuera").alias("Nacidos en el extranjero")))

periodo,Aumento con nacionalidad española,Nacidos en España,Nacidos en el extranjero
"""2002–2008""",1.285.345,915.216,370.129
"""2008–2014""",1.236.748,572.254,664.494
"""2014–2019""",248.797,-256.340,505.137
"""2019–2025""",148.137,-688.114,836.251


In [20]:
#| label: fig-explica
#| fig-cap: "Arriba izquierda: residentes con nacionalidad española nacidos fuera (% de los de nacionalidad española). Arriba derecha: residentes con nacionalidad extranjera nacidos en España (% de los de nacionalidad extranjera). Abajo izquierda: nacidos fuera con nacionalidad extranjera (% de los nacidos fuera). Abajo derecha: nacidos fuera con nacionalidad española (% de los nacidos fuera)"
fig_explica(cruce).show()

In [21]:
#| label: tbl-esp-ext-anual
#| tbl-cap: "Aumento anual de los residentes con nacionalidad española nacidos en el extranjero (personas y % sobre el año anterior)"
txt_anio(cruce.select("anio",
                      pl.col(ESP_EXT).diff().alias("Aumento anual (personas)"),
                      (100 * pl.col(ESP_EXT).pct_change()).round(1).alias("Aumento anual (%)")).drop_nulls())

año,Aumento anual (personas),Aumento anual (%)
"""2003""",45.650,"6,8"
"""2004""",48.582,"6,7"
"""2005""",56.060,"7,3"
"""2006""",59.014,"7,1"
"""2007""",76.885,"8,7"
"""2008""",83.938,"8,7"
"""2009""",90.488,"8,7"
"""2010""",82.117,"7,2"
"""2011""",124.960,"10,3"
"""2012""",108.365,"8,1"


In [22]:
#| label: fig-esp-ext-anual
#| fig-cap: "Aumento anual de los residentes con nacionalidad española nacidos en el extranjero (personas; al pasar el cursor, también en % sobre el año anterior)"
fig_esp_ext_anual(cruce).show()

**Lectura.** La tabla (@tbl-explica) y el gráfico (@fig-explica) descomponen la diferencia.

- **Quién explica la diferencia.** Entre 2002 y 2025 la diferencia aumenta en 1.956.113 personas: los residentes con nacionalidad española nacidos fuera suben 2.376.011 (de 675.371 a 3.051.382) y los de nacionalidad extranjera nacidos en España, 419.898 (de 79.245 a 499.143). *(Tabla, columnas de personas.)*
- **Nacionalidad española y nacidos fuera** (arriba izquierda): del 1,7 % de los residentes con nacionalidad española en 2002 al 7,2 % en 2025. Sube todos los años.
- **Nacionalidad extranjera y nacidos en España** (arriba derecha): del 4,6 % de los residentes con nacionalidad extranjera en 2002 al 10,8 % en 2018 (máximo) y al 7,2 % en 2025.
- **Nacidos fuera con nacionalidad española** (abajo derecha): 28,9 % en 2002, 17,8 % en 2008, 34,4 % en 2017 (máximo) y 32,2 % en 2025: uno de cada tres nacidos en el extranjero tiene nacionalidad española.
- **Residentes con nacionalidad española nacidos en España** (@tbl-esp-esp, @fig-esp-esp): pasan de 38.621.928 en 2002 (el 94,1 % del total) a su máximo de 40.130.756 en 2012, y bajan hasta 39.164.944 en 2025 (el 79,7 %). Son 965.812 menos que en 2012, pero 543.016 más que en 2002.
- **Quién sostiene el aumento de la población española** (@tbl-esp-composicion): los residentes con nacionalidad española nacidos en España suben 572.254 en 2008–2014 y bajan 256.340 en 2014–2019 y 688.114 en 2019–2025; los nacidos fuera suben 664.494, 505.137 y 836.251. En los dos últimos periodos el aumento de la población con nacionalidad española se debe por entero a los nacidos fuera.
- **Aumento reciente** (@tbl-esp-ext-anual, @fig-esp-ext-anual): los residentes con nacionalidad española nacidos fuera aumentaban en torno a 89.000–104.000 al año entre 2019 y 2022, y suben 132.075 en 2023, 199.654 en 2024 y 211.882 en 2025, los mayores aumentos de la serie en personas. En términos relativos no lo son: +7,6 % en 2024 y +7,5 % en 2025, frente a +12,1 % en 2014, +10,3 % en 2011 y +8,7 % en 2007–2009. Como el grupo es cada vez mayor, aumentos parecidos en % suponen más personas.

**Aprendizaje.** La diferencia entre definiciones no es un error de medida: es población con nacionalidad española que nació fuera, y crece cada año. Los datos no separan las causas (naturalizaciones, descendientes de españoles, retornos); la sección de explicaciones recoge las que documenta la literatura.

### 6. Un ejemplo: Sudamérica medida de dos maneras

**Qué se mira.** La misma región con las dos definiciones: residentes con nacionalidad de un país sudamericano y residentes nacidos en Sudamérica. **Cómo leer el gráfico:** si ambas líneas coincidieran, todos los nacidos en la región conservarían su nacionalidad de origen.

In [23]:
#| label: tbl-sudamerica
#| tbl-cap: "Sudamérica: residentes con nacionalidad sudamericana y residentes nacidos en Sudamérica"
txt_anio(pl.DataFrame({"anio": nac["anio"], "Con nacionalidad sudamericana": nac["Sudamérica"], "Nacidos en Sudamérica": pais["Sudamérica"]})
         .with_columns((pl.col("Nacidos en Sudamérica") - pl.col("Con nacionalidad sudamericana")).alias("Diferencia"))
         .filter(pl.col("anio").is_in(ANIOS_CLAVE)))

año,Con nacionalidad sudamericana,Nacidos en Sudamérica,Diferencia
"""2002""",603.294,780.432,177.138
"""2008""",1.525.069,1.981.303,456.234
"""2014""",907.577,1.834.604,927.027
"""2019""",891.421,2.147.783,1.256.362
"""2025""",1.958.005,3.739.859,1.781.854


In [24]:
#| label: fig-sudamerica
#| fig-cap: "Residentes con nacionalidad sudamericana y residentes nacidos en Sudamérica (millones de personas)"
fig_region_doble(nac, pais, "Sudamérica", "Con nacionalidad sudamericana", "Nacidos en Sudamérica").show()

**Lectura.** La tabla (@tbl-sudamerica) y el gráfico (@fig-sudamerica) aplican las dos definiciones a una región.

- **Con nacionalidad sudamericana:** 603.294 (2002), 1.525.069 (2008), 907.577 (2014; −40,5 % respecto a 2008), 891.421 (2019) y 1.958.005 (2025).
- **Nacidos en Sudamérica:** 780.432, 1.981.303, 1.834.604 (2014; −7,4 %), 2.147.783 y 3.739.859.
- **La caída de 2008–2014** es de 617.492 personas con la nacionalidad y de 146.699 con el lugar de nacimiento: menos de una cuarta parte.
- **La diferencia** crece de 177.138 a 1.781.854 personas. En 2025 equivale al 47,6 % de los nacidos en Sudamérica. Una parte son personas con doble nacionalidad, que el INE cuenta como españolas.

**Aprendizaje.** Tres cuartas partes de la «caída» sudamericana de 2008–2014 desaparecen al medir por lugar de nacimiento. Los datos no dicen cuántas de esas personas obtuvieron la nacionalidad española ni cuántas se fueron; solo que la población nacida en la región bajó mucho menos que la que conserva su nacionalidad.

::: {.callout-note collapse="true" title="Detalle técnico"}
**Definiciones y fórmulas.** Sea $P_{g,t}$ la población del grupo $g$ a 1 de enero del año $t$ y $P_t$ la del total.

- Cambio acumulado desde 2002: $100\,(P_{g,t}/P_{g,2002} - 1)$.
- Media anual en el periodo $[t_0, t_1]$: $(P_{g,t_1} - P_{g,t_0}) / (t_1 - t_0)$.
- Tasa anual compuesta: $\left(P_{g,t_1} / P_{g,t_0}\right)^{1/(t_1 - t_0)} - 1$.
- Peso: $100\,P_{g,t} / P_t$.
- Aportación al aumento del total: $(P_{g,t_1} - P_{g,t_0}) / (P_{t_1} - P_{t_0})$. Se cita solo cuando el total crece de forma apreciable; si es pequeño o negativo, el cociente se sale de 0–100 % y no es informativo.

**Tratamiento de los datos.** Solo población a 1 de enero, todos los sexos y edades. Las clasificaciones de Europa (UE28 hasta 2020 y UE27 desde 2020) se unifican en una sola región; en 2020 no se duplica ninguna región.

**Comprobaciones de calidad.** Española + Extranjera = Total; regiones de nacionalidad (con apátridas) = Extranjera; España + Extranjero = Total; regiones de nacimiento = Extranjero; los cuatro grupos del cruce suman el total. Diferencias máximas observadas: 1, 2, 1 y 1 personas (redondeo del INE); sin valores nulos tras unir Europa. El notebook completo (`_p01-analisis`) añade el análisis por país de nacimiento y una comprobación con bloques de 5 años.
:::

## Conclusiones

1. **La población crece por la migración.** Entre 2002 y 2025 la población residente aumenta en 8,1 millones. La población con nacionalidad extranjera explica el 63,9 % del aumento y la nacida en el extranjero, el 88,1 %.
2. **El crecimiento llega en dos impulsos.** +558.054 personas al año con nacionalidad extranjera en 2002–2008 y +343.535 en 2019–2025; entre ambos, −68.324 al año (2008–2014) y +34.882 (2014–2019).
3. **El origen de la población extranjera cambia.** Europa pasa del 34,0 % al 46,5 % (2014) y al 35,4 % (2025); Sudamérica, del 34,7 % al 19,4 % (2014) y al 28,3 %.
4. **La cifra de «extranjeros» depende de la definición.** En 2025 son el 14,1 % por nacionalidad y el 19,3 % por lugar de nacimiento; la diferencia (2.552.239 personas) se ha multiplicado por 4,3 desde 2002.
5. **La diferencia es población con nacionalidad española nacida fuera**, que crece cada año (de 675.371 a 3.051.382). Por eso en 2008–2014 la población con nacionalidad extranjera cae un 8,1 % y la nacida fuera sube un 1,3 %.

**Qué hay que recordar.** Antes de comparar o citar una cifra de «extranjeros», hay que saber si es por nacionalidad o por lugar de nacimiento: la primera mide un vínculo jurídico que cambia con los trámites; la segunda, un hecho que no cambia.

## Afirmaciones del debate público, contrastadas

Algunas afirmaciones recientes sobre la población extranjera pueden contrastarse con los resultados de este artículo. Cada veredicto remite a la tabla que lo respalda; los datos llegan al 1 de enero de 2025.

| Afirmación | Quién y cuándo | Veredicto | Qué dicen los resultados |
|---|---|---|---|
| España tiene «unos 47 millones de habitantes» | Rocío de Meer (Vox), 7-VII-2025 [@maldita_vox_2025; @thelocal_2025] | **Falsa** | 49.128.297 residentes a 1-I-2025; más de 47 millones desde 2020 (47.318.050) (@tbl-serie-anual). |
| Unos «ocho millones de personas» que «tendrán que volver a sus países» | Rocío de Meer (Vox), 7-VII-2025 [@efe_vox_2025] | **Choca con los datos** | Con nacionalidad extranjera hay 6.911.971 personas, y 499.143 de ellas nacieron en España (@tbl-resumen-nac, @tbl-explica). De los 9.464.210 nacidos en el extranjero, 3.051.382 (el 32,2 %) tienen nacionalidad española (@tbl-definiciones, @tbl-explica). |
| El Gobierno busca la «sustitución demográfica del pueblo español» | Moción de Vox en el Congreso, 10-XII-2024 [@eldebate_vox_2024] | **Parcialmente cierta en lo descriptivo; la intención no es comprobable** | Los residentes con nacionalidad española nacidos en España pasan del 94,1 % al 79,7 % de la población y son 965.812 menos que en 2012, pero 543.016 más que en 2002 (@tbl-esp-esp). |
| En los últimos cuatro años «han llegado otros 1,5 millones de inmigrantes» | Alberto Núñez Feijóo (PP), 2-X-2025 [@ara_feijoo_2025] | **Cierta como aumento neto por nacionalidad** | Entre 2021 y 2025: +1.509.269 con nacionalidad extranjera; +2.209.413 nacidos en el extranjero (@tbl-serie-anual). Las llegadas brutas son mayores que el aumento neto. |
| «Uno de cada cinco» ha nacido en el extranjero | Alberto Núñez Feijóo (PP), 2-X-2025 [@ara_feijoo_2025] | **Cierta** | El 19,3 % en 2025; con nacionalidad extranjera, el 14,1 % (uno de cada siete) (@tbl-definiciones). |
| Los inmigrantes impulsan el crecimiento de la población | Elma Saiz (Gobierno), diciembre de 2025 [@theobjective_saldo_2025] | **Cierta para España** | La población nacida en el extranjero explica el 88,1 % del aumento de 2002–2025; la de nacionalidad extranjera, el 63,9 % (@tbl-definiciones-periodos). La comparación con otros países no la cubren estos datos. |

**No comprobables con estos datos.** «3,7 millones de inmigrantes más desde que gobierna Sánchez» (difundida en marzo de 2026 [@maldita_37_2026]): entre 2018 y 2025 el aumento es de 2.334.649 por nacionalidad y de 3.256.701 por lugar de nacimiento (@tbl-serie-anual); con datos de 2026, que no se analizan aquí, podría acercarse a esa cifra por lugar de nacimiento. «2,5 millones traídos en los últimos dos años» (Santiago Abascal, 20-V-2026 [@theobjective_abascal_2026]) se refiere a un periodo posterior a estos datos y a llegadas, no a población. Las afirmaciones sobre llegadas irregulares, delincuencia, empleo o ayudas necesitan otras fuentes.

**Qué hay que recordar.** Dos de estas afirmaciones, hechas por la misma persona en el mismo acto, usan definiciones distintas sin decirlo: el aumento de 1,5 millones es por nacionalidad y el «uno de cada cinco», por lugar de nacimiento. Es el mismo aprendizaje del artículo: sin la definición, la cifra no se puede contrastar.

## Explicaciones y marco teórico

Los datos describen; lo que sigue es interpretación apoyada en la literatura, y se indica qué parte de los resultados es compatible con cada explicación (no se demuestra con estos datos).

- **El ritmo depende del ciclo económico.** Para España, Lacuesta y Puente (2010) estiman que las entradas de inmigrantes responden al crecimiento del PIB per cápita (alrededor de un 10 % menos de entradas por cada punto menos de crecimiento) y esperan que, en una recesión, la población inmigrante crezca menos sobre todo por menos llegadas [@lacuesta_puente_2010]. Es compatible con el salto de 2002–2008 y el estancamiento de 2008–2014. Los datos son de población, no de flujos: no permiten separar llegadas y salidas.
- **La nacionalidad se adquiere, y por vías distintas según el origen.** El Código Civil exige con carácter general diez años de residencia legal y continuada; bastan cinco para refugiados y dos para nacionales de América Central y del Sur, Andorra, Filipinas, Guinea Ecuatorial, Portugal y sefardíes [@finotelli_labarbera_2017]. Entre 2005 y 2015 se naturalizaron más de un millón de extranjeros, el 74 % de América Central y del Sur, y la mayoría después de 2008 [@finotelli_labarbera_2017]. Es compatible con que en 2008–2014 los residentes con nacionalidad española nacidos fuera aumenten en 664.494 (@tbl-esp-composicion), con que la población con nacionalidad extranjera caiga mientras la nacida fuera se mantiene, y con el caso de Sudamérica. Cada naturalización pasa a una persona de «extranjera» a «española» sin que nadie se mueva.
- **La nacionalidad también se reconoce por descendencia.** La disposición adicional octava de la Ley 20/2022 permite optar a la nacionalidad española de origen a nacidos fuera de España de padre, madre, abuelo o abuela originariamente españoles que la perdieron o renunciaron a ella por exilio, a hijos de españolas que la perdieron por casarse con extranjeros antes de la Constitución de 1978 y a hijos mayores de edad de quienes ya la obtuvieron por esa vía o por la disposición adicional séptima de la Ley 52/2007, con un plazo de dos años desde la entrada en vigor (21 de octubre de 2022), prorrogable un año [@ley_20_2022; @instruccion_2022]. El aumento anual de residentes con nacionalidad española nacidos fuera se duplica en personas entre 2022 y 2025 (de 103.752 a 211.882) y pasa del 4,3 % al 7,5 % en términos relativos (@tbl-esp-ext-anual). Coincide en el tiempo con esa ley (@fig-esp-ext-anual), pero estos datos no permiten atribuirle el aumento. La opción no exige residir en España y puede presentarse en las oficinas consulares del Registro Civil [@instruccion_2022], así que quien obtiene la nacionalidad por esta vía solo aparece en estos datos si reside en España; el aumento incluye además naturalizaciones, retornos y otras vías, y antes de la ley ya hubo aumentos relativos mayores (+12,1 % en 2014).
- **Cada definición mide algo distinto.** El lugar de nacimiento es un hecho que no cambia, así que es un criterio más estable para identificar a la primera generación de inmigrantes, pero no capta a los hijos de inmigrantes nacidos en España: ni a los 499.143 que en 2025 tienen nacionalidad extranjera ni a quienes ya tienen la española. La nacionalidad refleja la situación jurídica actual. Elegir una u otra cambia los resultados: el Banco de España, por ejemplo, calcula la contribución de la población extranjera al crecimiento del PIB per cápita en 2022–2024 con las dos y obtiene 0,4 puntos (nacionalidad) y 0,7 (lugar de nacimiento) [@cuadrado_regil_2025].
- **Sin un saldo migratorio positivo, la población bajaría.** Los nacimientos (318.005 en 2024) son menos que las defunciones (436.118): el crecimiento vegetativo es negativo desde 2017 [@ine_mnp_2024]. Como la variación de la población es la suma del crecimiento vegetativo y del saldo migratorio, todo el aumento reciente procede de la migración. En estos datos, los residentes con nacionalidad española nacidos en España bajan 256.340 en 2014–2019 y 688.114 en 2019–2025 (@tbl-esp-composicion).

## Implicaciones

Estos datos no miden efectos; las implicaciones se apoyan en estudios que sí lo hacen, y se señala su alcance y sus límites.

1. **Para citar y comparar cifras.** Indicar siempre la definición («con nacionalidad extranjera» o «nacidos en el extranjero»). Con una u otra, la población extranjera de 2025 difiere en 2,55 millones y la evolución de la crisis cambia de signo. El lugar de nacimiento es el criterio más estable en el tiempo; la nacionalidad es el relevante cuando importa la situación jurídica (derechos, permisos, voto).
2. **Demografía y actividad económica.** Con un crecimiento vegetativo negativo desde 2017 [@ine_mnp_2024], la evolución de la población depende del saldo migratorio. El Banco de España estima que la población extranjera aportó entre 0,4 y 0,7 puntos al crecimiento medio anual del PIB per cápita de 2022–2024 (2,9 %), sobre todo por la tasa de empleo; es una descomposición mecánica que no modela efectos indirectos sobre los nacionales [@cuadrado_regil_2025]. No se puede extrapolar a otros periodos.
3. **Empleo y salarios de la población ya residente.** La evidencia no es unánime. Para España en la segunda mitad de los noventa, Carrasco, Jimeno y Ortega no encuentran un efecto negativo significativo de la inmigración sobre el empleo y los salarios de los nativos [@carrasco_2008]. Para Estados Unidos, Borjas estima que un aumento de la oferta del 10 % reduce un 3–4 % los salarios de los trabajadores que compiten con los inmigrantes [@borjas_2003]. Dustmann, Schönberg y Stuhler muestran que parte de la discrepancia procede de los supuestos del modelo y de si se estima un efecto relativo o total [@dustmann_2016]. Lo prudente es evaluar cada caso: depende de en qué ocupaciones se concentra la llegada y de a quién sustituye o complementa.
4. **Finanzas públicas.** El efecto fiscal neto de la inmigración es, por término medio en la OCDE, cercano a cero, con diferencias entre países que dependen sobre todo del empleo y de la edad de los inmigrantes [@oecd_2013]. Según ese estudio, el empleo es el factor que más pesa en la aportación fiscal neta de los inmigrantes.

**Lo que estos datos no permiten concluir.** Cuánta de la variación se debe a llegadas, salidas, nacimientos, defunciones o cambios de nacionalidad; ni qué efecto tiene la migración sobre salarios, precios o servicios públicos; ni la situación administrativa de las personas, que la ECP no distingue.

## Limitaciones

- Son **stocks, no flujos**: la variación mezcla llegadas, salidas, nacimientos, defunciones y cambios de nacionalidad.
- Los **cambios de nacionalidad** pasan población de «extranjera» a «española» sin que nadie se mueva; el análisis no los puede cuantificar.
- «Nacido en el extranjero» incluye a españoles nacidos fuera; no equivale a «llegada reciente».
- Las personas con doble nacionalidad (española y otra) se cuentan como españolas, lo que reduce la cifra de extranjeros por nacionalidad.
- Las regiones agregan países muy distintos (por ejemplo, «Europa» incluye a la UE y al resto del continente); el análisis no baja a países.
- Posible **ruptura de serie** alrededor de 2021 (transición a la ECP y revisión censal) y cambio de clasificación de Europa en 2020. En el total no hay un salto aislado en 2020–2022, pero el aumento de 2023 (+598.634, @tbl-serie-anual) es el mayor desde 2008 y, con estos datos, no se puede descartar que parte refleje cambios de método o de registro.
- La ECP mide personas inscritas como residentes: depende de cómo y cuándo se registran altas y bajas.
- Datos hasta el 1-I-2025; solo nivel nacional; sin desagregar por edad ni sexo.
- Análisis **descriptivo**: las explicaciones son interpretación, no resultado de estos datos.

## Referencias

<!-- Correcciones: AAAA-MM-DD · qué se corrigió -->